<a href="https://colab.research.google.com/github/cleberte88/comissoes_vendedores/blob/main/comissaoExternos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTAÇÃO DAS BIBLIOTECAS NECESSÁRIAS PRO PROJETO
from pyspark.sql import SparkSession as ss
from pyspark.sql import functions as f
from pyspark.sql.types import DoubleType
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
!pip install xlsxwriter

In [ ]:
spark = ss.builder.appName("comissoesExternos").getOrCreate()

In [ ]:
url_com_rec = "/content/drive/MyDrive/comissoes_externos/comissoes_recebidos.csv"
url_dev_rec = "/content/drive/MyDrive/comissoes_externos/comissoes_recebidos_dev.csv"
url_dev_dev_post = "/content/drive/MyDrive/comissoes_externos/comissao_recebidos_dev_post.csv"
url_dev_geral = "/content/drive/MyDrive/comissoes_externos/Indicador_devolucao.csv"

df_com_rec = spark.read.csv(
    url_com_rec,
    sep=",",
    header=True,
    inferSchema=True
)
df_dev_rec = spark.read.csv(
    url_dev_rec,
    sep=",",
    header=True,
    inferSchema=True
)
df_dev_rec_post = spark.read.csv(
    url_dev_dev_post,
    sep=",",
    header=True,
    inferSchema=True
)
df_indicador_dev = spark.read.csv(
    url_dev_geral,
    sep=",",
    header=True,
    inferSchema=True
)


In [ ]:
df_com_rec.show(1, False)

+-------+-----------------+----------------------------------------+-------------+-------------+----------+-------------+--------+-------------+-------------+-------------------+--------------------+-------------------------+-------------------------------+-----------------+-------------------+------------------------------------+
|Nº CR  |Código do Cliente|Nome do Cliente                         |Data da Baixa|Nº Interno NF|NotaFiscal|Valor Baixado|Comissão|Parcela Atual|Parcela Total|Valor Pago Comissão|DocumentoRenegociado|Comissão Paga Renegociada|Valor Pago Comissão Renegociada|Representante    |Data da Nota Fiscal|Comments                            |
+-------+-----------------+----------------------------------------+-------------+-------------+----------+-------------+--------+-------------+-------------+-------------------+--------------------+-------------------------+-------------------------------+-----------------+-------------------+------------------------------------+
|

In [ ]:
df_com_rec = df_com_rec.withColumnsRenamed({
    "Nº CR":"num_cr",
    "Código do Cliente":"codigo_cliente",
    "Nome do Cliente":"nome_cliente",
    "Data da Baixa":"data_baixa",
    "Nº Interno NF":"df_saida",
    "Valor Baixado":"valor_baixado",
    "Parcela Atual":"parcela_atual",
    "Parcela Total":"parcela_total",
    "Valor Pago Comissão":"valor_pago_comissao",
    "Comissão Paga Renegociada":"comissao_paga_reneg",
    "Valor Pago Comissão Renegociada":"valor_pago_com_reneg",
    "Data da Nota Fiscal":"data_nf"
})

In [ ]:
df_com_rec.printSchema()

root
 |-- num_cr: integer (nullable = true)
 |-- codigo_cliente: string (nullable = true)
 |-- nome_cliente: string (nullable = true)
 |-- data_baixa: string (nullable = true)
 |-- df_saida: integer (nullable = true)
 |-- NotaFiscal: integer (nullable = true)
 |-- valor_baixado: double (nullable = true)
 |-- Comissão: double (nullable = true)
 |-- parcela_atual: integer (nullable = true)
 |-- parcela_total: integer (nullable = true)
 |-- valor_pago_comissao: double (nullable = true)
 |-- DocumentoRenegociado: integer (nullable = true)
 |-- comissao_paga_reneg: double (nullable = true)
 |-- valor_pago_com_reneg: double (nullable = true)
 |-- Representante: string (nullable = true)
 |-- data_nf: string (nullable = true)
 |-- Comments: string (nullable = true)



In [ ]:
df_com_rec = df_com_rec.withColumn("data_baixa", f.to_date(df_com_rec["data_baixa"], "MM/dd/yyyy"))\
          .withColumn("data_nf", f.to_date(df_com_rec["data_nf"], "MM/dd/yyyy"))

In [ ]:
df_com_rec.printSchema()

root
 |-- num_cr: integer (nullable = true)
 |-- codigo_cliente: string (nullable = true)
 |-- nome_cliente: string (nullable = true)
 |-- data_baixa: date (nullable = true)
 |-- df_saida: integer (nullable = true)
 |-- NotaFiscal: integer (nullable = true)
 |-- valor_baixado: double (nullable = true)
 |-- Comissão: double (nullable = true)
 |-- parcela_atual: integer (nullable = true)
 |-- parcela_total: integer (nullable = true)
 |-- valor_pago_comissao: double (nullable = true)
 |-- DocumentoRenegociado: integer (nullable = true)
 |-- comissao_paga_reneg: double (nullable = true)
 |-- valor_pago_com_reneg: double (nullable = true)
 |-- Representante: string (nullable = true)
 |-- data_nf: date (nullable = true)
 |-- Comments: string (nullable = true)



In [ ]:
df_com_rec = df_com_rec.drop("data_nd")

In [ ]:
df_com_rec.show(1, False)

+-------+--------------+----------------------------------------+----------+--------+----------+-------------+--------+-------------+-------------+-------------------+--------------------+-------------------+--------------------+-----------------+----------+------------------------------------+
|num_cr |codigo_cliente|nome_cliente                            |data_baixa|df_saida|NotaFiscal|valor_baixado|Comissão|parcela_atual|parcela_total|valor_pago_comissao|DocumentoRenegociado|comissao_paga_reneg|valor_pago_com_reneg|Representante    |data_nf   |Comments                            |
+-------+--------------+----------------------------------------+----------+--------+----------+-------------+--------+-------------+-------------+-------------------+--------------------+-------------------+--------------------+-----------------+----------+------------------------------------+
|2729771|CL018618      |50.943.456 ALINE EVELYN GAMA DE CARVALHO|2026-02-11|1264690 |1233153   |94.16        |0.

In [ ]:
df_com_rec.count()

40358

In [ ]:
df_com_rec.createOrReplaceTempView("v_com_rec")

In [ ]:
spark.sql(
    """
    SELECT cr.*,
          CASE
            WHEN cr.Representante LIKE '%MAR E RIO%' THEN "N"
            ELSE cr.Representante
          END AS RepValida,
          (cr.valor_pago_comissao + cr.valor_pago_com_reneg) AS valor_pago,
          CASE
            WHEN valor_pago = 0 THEN LEAST(cr.valor_pago_comissao, cr.valor_pago_com_reneg)
            ELSE valor_pago
          END AS Id,
          "A" AS REP,
          "A" AS CORRETO,
          CONCAT(cr.codigo_cliente, cr.NotaFiscal) AS cl_nf,
          CONCAT(cr.codigo_cliente, cr.df_saida, cr.valor_baixado) AS cl_nf_valorBaixado,
          CONCAT(cr.codigo_cliente, cr.df_saida, cr.valor_pago_comissao) AS cl_nf_valorPago
    FROM v_com_rec cr
    """
).show(2, False)

+-------+--------------+----------------------------------------+----------+--------+----------+-------------+--------+-------------+-------------+-------------------+--------------------+-------------------+--------------------+-----------------+----------+------------------------------------+-----------------+----------+----+---+-------+---------------+--------------------+-------------------+
|num_cr |codigo_cliente|nome_cliente                            |data_baixa|df_saida|NotaFiscal|valor_baixado|Comissão|parcela_atual|parcela_total|valor_pago_comissao|DocumentoRenegociado|comissao_paga_reneg|valor_pago_com_reneg|Representante    |data_nf   |Comments                            |RepValida        |valor_pago|Id  |REP|CORRETO|cl_nf          |cl_nf_valorBaixado  |cl_nf_valorPago    |
+-------+--------------+----------------------------------------+----------+--------+----------+-------------+--------+-------------+-------------+-------------------+--------------------+--------------

In [ ]:
df_com_rec = spark.sql(
    """
    SELECT cr.*,
          CASE
            WHEN cr.Representante LIKE '%MAR E RIO%' THEN "N"
            ELSE cr.Representante
          END AS RepValida,
          (cr.valor_pago_comissao + cr.valor_pago_com_reneg) AS valor_pago,
          CASE
            WHEN valor_pago = 0 THEN LEAST(cr.valor_pago_comissao, cr.valor_pago_com_reneg)
            ELSE valor_pago
          END AS Id,
          "A" AS REP,
          "A" AS CORRETO,
          CONCAT(cr.codigo_cliente, cr.NotaFiscal) AS cl_nf,
          CONCAT(cr.codigo_cliente, cr.df_saida, cr.valor_baixado) AS cl_nf_valorBaixado,
          CONCAT(cr.codigo_cliente, cr.df_saida, cr.valor_pago_comissao) AS cl_nf_valorPago
    FROM v_com_rec cr
    """
)

In [ ]:
df_com_rec.count()

40358

In [ ]:
df_com_rec.createOrReplaceTempView("v_com_rec")

In [ ]:
#=====================================================

df_dev_rec = df_dev_rec.toDF(
    "cardcode",
    "cardname",
    "NumDev",
    "NFDev",
    "data_dev",
    "N_interno_nf_saida",
    "nota_fiscal",
    "status_devolucao",
    "valor_total_dev",
    "Representante_A",   # primeiro Representante
    "cod_cliente",
    "nome_cliente",
    "NF",
    "Comissao",
    "valor_pago_comissao",
    "Representante_B",   # segundo Representante
    "data_nf",
    "Comments"
)


In [ ]:
df_com_rec.count()

40358

In [ ]:
# Daqui pra baixo vou tratar as devoluções para depois voltar pras vendas
# novamente para conferir as demais colunas faltantes.

df_dev_rec.show(1, False)

+--------+---------------------------+------+-----+----------+------------------+-----------+----------------+---------------+------------------+-----------+---------------------------+-------+--------+-------------------+------------------+----------+------------------------------------+
|cardcode|cardname                   |NumDev|NFDev|data_dev  |N_interno_nf_saida|nota_fiscal|status_devolucao|valor_total_dev|Representante_A   |cod_cliente|nome_cliente               |NF     |Comissao|valor_pago_comissao|Representante_B   |data_nf   |Comments                            |
+--------+---------------------------+------+-----+----------+------------------+-----------+----------------+---------------+------------------+-----------+---------------------------+-------+--------+-------------------+------------------+----------+------------------------------------+
|CL002582|RESTAURANTE MINAS TCHE LTDA|39641 |797  |02/06/2026|2021560           |180512     |Devolução Total |1835.22        |MSC 

In [ ]:
df_dev_rec.printSchema()

root
 |-- cardcode: string (nullable = true)
 |-- cardname: string (nullable = true)
 |-- NumDev: integer (nullable = true)
 |-- NFDev: integer (nullable = true)
 |-- data_dev: string (nullable = true)
 |-- N_interno_nf_saida: integer (nullable = true)
 |-- nota_fiscal: integer (nullable = true)
 |-- status_devolucao: string (nullable = true)
 |-- valor_total_dev: double (nullable = true)
 |-- Representante_A: string (nullable = true)
 |-- cod_cliente: string (nullable = true)
 |-- nome_cliente: string (nullable = true)
 |-- NF: integer (nullable = true)
 |-- Comissao: double (nullable = true)
 |-- valor_pago_comissao: double (nullable = true)
 |-- Representante_B: string (nullable = true)
 |-- data_nf: string (nullable = true)
 |-- Comments: string (nullable = true)



In [ ]:
df_dev_rec = df_dev_rec.withColumnsRenamed({
    "Data Dev":"data_dev",
    "Número Interno Nota Fiscal de Saída":"N_interno_nf_saida",
    "Nota Fiscal":"nota_fiscal",
    "Status Devolução":"status_devolucao",
    "Valor Total Devolução":"valor_total_dev",
    "Código do Cliente":"cod_cliente",
    "Nome do Cliente":"nome_cliente",
    "Nº Interno NF":"NF",
    "Valor Pago Comissão":"valor_pago_comissao",
    "Data da Nota Fiscal":"data_nf"
})

In [ ]:
df_dev_rec.printSchema()

root
 |-- cardcode: string (nullable = true)
 |-- cardname: string (nullable = true)
 |-- NumDev: integer (nullable = true)
 |-- NFDev: integer (nullable = true)
 |-- data_dev: string (nullable = true)
 |-- N_interno_nf_saida: integer (nullable = true)
 |-- nota_fiscal: integer (nullable = true)
 |-- status_devolucao: string (nullable = true)
 |-- valor_total_dev: double (nullable = true)
 |-- Representante_A: string (nullable = true)
 |-- cod_cliente: string (nullable = true)
 |-- nome_cliente: string (nullable = true)
 |-- NF: integer (nullable = true)
 |-- Comissao: double (nullable = true)
 |-- valor_pago_comissao: double (nullable = true)
 |-- Representante_B: string (nullable = true)
 |-- data_nf: string (nullable = true)
 |-- Comments: string (nullable = true)



In [ ]:
df_dev_rec = df_dev_rec.withColumn("data_dev", f.to_date(df_dev_rec["data_dev"], "MM/dd/yyyy"))\
          .withColumn("data_nf", f.to_date(df_dev_rec["data_nf"], "MM/dd/yyyy"))

In [ ]:
df_dev_rec.createOrReplaceTempView("v_dev_rec")

In [ ]:
spark.sql(
    """
    SELECT dr.*,
          CASE
            WHEN dr.Representante_B LIKE '%MAR E RIO%'
              THEN "N"
            ELSE dr.Representante_B
          END AS RepValida,
          CONCAT(dr.NF, dr.Comissao) AS Valida,
          COUNT(*) OVER (PARTITION BY dr.cardcode, dr.N_interno_nf_saida,
          valor_total_dev, dr.data_nf) AS Repete,
          CONCAT(dr.cardcode, dr.N_interno_nf_saida, dr.valor_pago_comissao) AS cod_nf_valorPago
    FROM v_dev_rec dr
    """
).show(1, False)

+--------+---------------------------+------+-----+----------+------------------+-----------+----------------+---------------+------------------+-----------+---------------------------+-------+--------+-------------------+------------------+----------+------------------------------------+------------------+-------------+------+--------------------+
|cardcode|cardname                   |NumDev|NFDev|data_dev  |N_interno_nf_saida|nota_fiscal|status_devolucao|valor_total_dev|Representante_A   |cod_cliente|nome_cliente               |NF     |Comissao|valor_pago_comissao|Representante_B   |data_nf   |Comments                            |RepValida         |Valida       |Repete|cod_nf_valorPago    |
+--------+---------------------------+------+-----+----------+------------------+-----------+----------------+---------------+------------------+-----------+---------------------------+-------+--------+-------------------+------------------+----------+------------------------------------+---------

In [ ]:
df_dev_rec = spark.sql(
    """
    SELECT dr.*,
          CASE
            WHEN dr.Representante_B LIKE '%MAR E RIO%'
              THEN "N"
            ELSE dr.Representante_B
          END AS RepValida,
          CONCAT(dr.NF, dr.Comissao) AS Valida,
          COUNT(*) OVER (PARTITION BY dr.cardcode, dr.N_interno_nf_saida,
          valor_total_dev, dr.data_nf) AS Repete,
          CONCAT(dr.cardcode, dr.N_interno_nf_saida, dr.valor_pago_comissao) AS cod_nf_valorPago
    FROM v_dev_rec dr
    """
)

In [ ]:
df_dev_rec.createOrReplaceTempView("v_dev_rec")

In [ ]:
# A partir daqui vamos configurar as devoluções posteriores ao recebimento.

df_dev_rec_post.show(1, False)

+--------+-----------------------------+------+-----+----------+-----------------------------------+-----------+----------------+---------------------+--------------+-------+-----------------+-----------------------------+-------------+-------------+-------------+---------+-------------------+---------------+-------------------+------------------------------------+
|cardcode|cardname                     |NumDev|NFDev|Data Dev  |Número Interno Nota Fiscal de Saída|Nota Fiscal|Status Devolução|Valor Total Devolução|Representante9|Nº CR  |Código do Cliente|Nome do Cliente              |Data da Baixa|Nº Interno NF|Valor Baixado|Comissão |Valor Pago Comissão|Representante18|Data da Nota Fiscal|Comments                            |
+--------+-----------------------------+------+-----+----------+-----------------------------------+-----------+----------------+---------------------+--------------+-------+-----------------+-----------------------------+-------------+-------------+-------------+

In [ ]:
df_dev_rec_post.printSchema()

root
 |-- cardcode: string (nullable = true)
 |-- cardname: string (nullable = true)
 |-- NumDev: integer (nullable = true)
 |-- NFDev: integer (nullable = true)
 |-- Data Dev: string (nullable = true)
 |-- Número Interno Nota Fiscal de Saída: integer (nullable = true)
 |-- Nota Fiscal: integer (nullable = true)
 |-- Status Devolução: string (nullable = true)
 |-- Valor Total Devolução: double (nullable = true)
 |-- Representante9: string (nullable = true)
 |-- Nº CR: integer (nullable = true)
 |-- Código do Cliente: string (nullable = true)
 |-- Nome do Cliente: string (nullable = true)
 |-- Data da Baixa: string (nullable = true)
 |-- Nº Interno NF: integer (nullable = true)
 |-- Valor Baixado: double (nullable = true)
 |-- Comissão: double (nullable = true)
 |-- Valor Pago Comissão: double (nullable = true)
 |-- Representante18: string (nullable = true)
 |-- Data da Nota Fiscal: string (nullable = true)
 |-- Comments: string (nullable = true)



In [ ]:
df_dev_rec_post = df_dev_rec_post.withColumnsRenamed({
    "Data Dev":"data_dev",
    "Número Interno Nota Fiscal de Saída":"N_interno_nf_saida",
    "Nota Fiscal":"nota_fiscal",
    "Status Devolução":"status_devolucao",
    "Valor Total Devolução":"valor_total_dev",
    "Representante9":"Representante_A",
    "Nº CR":"Num_CR",
    "Código do Cliente":"cod_cliente",
    "Nome do Cliente":"nome_cliente",
    "Data da Baixa":"data_baixa",
    "Nº Interno NF":"NF",
    "Valor Baixado":"valor_baixado",
    "Valor Pago Comissão":"valor_pago_comissao",
    "Data da Nota Fiscal":"data_nf"
})
df_dev_rec_post.show(1, False)

+--------+-----------------------------+------+-----+----------+------------------+-----------+----------------+---------------+---------------+-------+-----------+-----------------------------+----------+-------+-------------+---------+-------------------+---------------+----------+------------------------------------+
|cardcode|cardname                     |NumDev|NFDev|data_dev  |N_interno_nf_saida|nota_fiscal|status_devolucao|valor_total_dev|Representante_A|Num_CR |cod_cliente|nome_cliente                 |data_baixa|NF     |valor_baixado|Comissão |valor_pago_comissao|Representante18|data_nf   |Comments                            |
+--------+-----------------------------+------+-----+----------+------------------+-----------+----------------+---------------+---------------+-------+-----------+-----------------------------+----------+-------+-------------+---------+-------------------+---------------+----------+------------------------------------+
|CL009879|AMANDA FERREIRA DE SOUZA

In [ ]:
df_dev_rec_post = df_dev_rec_post.withColumn("data_dev", f.to_date(df_dev_rec_post["data_dev"], "MM/dd/yyyy"))\
               .withColumn("data_baixa", f.to_date(df_dev_rec_post["data_baixa"], "MM/dd/yyyy"))\
               .withColumn("data_nf", f.to_date(df_dev_rec_post["data_nf"], "MM/dd/yyyy"))

In [ ]:
df_dev_rec_post.createOrReplaceTempView("v_dev_post")

In [ ]:
spark.sql(
    """
    SELECT dp.*,
          CASE
            WHEN dp.Representante18 LIKE '%MAR E RIO%'
              THEN "N"
            ELSE dp.Representante18
          END AS RepValida,
          CONCAT(dp.NF, dp.valor_pago_comissao) AS dev
    FROM v_dev_post dp
    """
).show(1, False)

+--------+-----------------------------+------+-----+----------+------------------+-----------+----------------+---------------+---------------+-------+-----------+-----------------------------+----------+-------+-------------+---------+-------------------+---------------+----------+------------------------------------+-------------+-----------+
|cardcode|cardname                     |NumDev|NFDev|data_dev  |N_interno_nf_saida|nota_fiscal|status_devolucao|valor_total_dev|Representante_A|Num_CR |cod_cliente|nome_cliente                 |data_baixa|NF     |valor_baixado|Comissão |valor_pago_comissao|Representante18|data_nf   |Comments                            |RepValida    |dev        |
+--------+-----------------------------+------+-----+----------+------------------+-----------+----------------+---------------+---------------+-------+-----------+-----------------------------+----------+-------+-------------+---------+-------------------+---------------+----------+--------------------

In [ ]:
df_dev_rec_post = spark.sql(
    """
    SELECT dp.*,
          CASE
            WHEN dp.Representante18 LIKE '%MAR E RIO%'
              THEN "N"
            ELSE dp.Representante18
          END AS RepValida,
          CONCAT(dp.NF, dp.valor_pago_comissao) AS dev
    FROM v_dev_post dp
    """
)

In [ ]:
df_dev_rec_post.createOrReplaceTempView("v_dev_post")

In [ ]:
# Aqui já vou começar agora as DEV GERAIS

df_indicador_dev.show(1, False)

+-----+------------------+----------+-------------+-----------------+-------+----------+--------------------+-----------+---------+----------------+------------+-----------------------------------------+-----------------+----+-------------------+-------------+-----------+---------+----------+---------------+---------+-------------+
|Carga|Rota              |Data Nfe. |Nº Nota Saída|Nº Nota Devolução|Valor  |Data Dev. |Data Lancamento Dev.|Hora       |Qtd. Nota|Faturista       |Cod. Cliente|Nome Cliente                             |Tipo Devolução   |Cod.|Motivo Cancelamento|Representante|Setor Resp.|Nº Viagem|Fornecedor|Nome Fornecedor|Motorista|Placa Veiculo|
+-----+------------------+----------+-------------+-----------------+-------+----------+--------------------+-----------+---------+----------------+------------+-----------------------------------------+-----------------+----+-------------------+-------------+-----------+---------+----------+---------------+---------+-------------

In [ ]:
df_indicador_dev.withColumnsRenamed({
    "Data Nfe.":"data_nf",
    "Nº Nota Saída":"N_nf_saida",
    "Nº Nota Devolução":"nota_dev",
    "Data Dev.":"data_dev",
    "Data Lancamento Dev.":"data_lancamento_dev",
    "Qtd. Nota":"qtd_nota",
    "Cod. Cliente":"cod_cliente",
    "Nome Cliente":"nome_cliente",
    "Tipo Devolução":"tipo_dev",
    "Cod.":"cod",
    "Motivo Cancelamento":"motivo_cancelamento",
    "Setor Resp.":"setor_resp",
    "Nº Viagem":"num_viagem",
    "Nome Fornecedor":"nome_fornecedor",
    "Placa Veiculo":"placa_veiculo"
}).show(1, False)

+-----+------------------+----------+----------+--------+-------+----------+-------------------+-----------+--------+----------------+-----------+-----------------------------------------+-----------------+---+-------------------+-------------+----------+----------+----------+---------------+---------+-------------+
|Carga|Rota              |data_nf   |N_nf_saida|nota_dev|Valor  |data_dev  |data_lancamento_dev|Hora       |qtd_nota|Faturista       |cod_cliente|nome_cliente                             |tipo_dev         |cod|motivo_cancelamento|Representante|setor_resp|num_viagem|Fornecedor|nome_fornecedor|Motorista|placa_veiculo|
+-----+------------------+----------+----------+--------+-------+----------+-------------------+-----------+--------+----------------+-----------+-----------------------------------------+-----------------+---+-------------------+-------------+----------+----------+----------+---------------+---------+-------------+
|NULL |SP-RIO PRETO NORTE|07/28/2023|3066     

In [ ]:
df_indicador_dev = df_indicador_dev.withColumnsRenamed({
    "Data Nfe.":"data_nf",
    "Nº Nota Saída":"N_nf_saida",
    "Nº Nota Devolução":"nota_dev",
    "Data Dev.":"data_dev",
    "Data Lancamento Dev.":"data_lancamento_dev",
    "Qtd. Nota":"qtd_nota",
    "Cod. Cliente":"cod_cliente",
    "Nome Cliente":"nome_cliente",
    "Tipo Devolução":"tipo_dev",
    "Cod.":"cod",
    "Motivo Cancelamento":"motivo_cancelamento",
    "Setor Resp.":"setor_resp",
    "Nº Viagem":"num_viagem",
    "Nome Fornecedor":"nome_fornecedor",
    "Placa Veiculo":"placa_veiculo"
})

In [ ]:
df_indicador_dev.withColumn("data_nf", f.to_date(df_indicador_dev["data_nf"], "MM/dd/yyyy"))\
                .withColumn("data_dev", f.to_date(df_indicador_dev["data_dev"], "MM/dd/yyyy"))\
                .withColumn("data_lancamento_dev", f.to_date(df_indicador_dev["data_lancamento_dev"], "MM/dd/yyyy")).show(1, False)

+-----+------------------+----------+----------+--------+-------+----------+-------------------+-----------+--------+----------------+-----------+-----------------------------------------+-----------------+---+-------------------+-------------+----------+----------+----------+---------------+---------+-------------+
|Carga|Rota              |data_nf   |N_nf_saida|nota_dev|Valor  |data_dev  |data_lancamento_dev|Hora       |qtd_nota|Faturista       |cod_cliente|nome_cliente                             |tipo_dev         |cod|motivo_cancelamento|Representante|setor_resp|num_viagem|Fornecedor|nome_fornecedor|Motorista|placa_veiculo|
+-----+------------------+----------+----------+--------+-------+----------+-------------------+-----------+--------+----------------+-----------+-----------------------------------------+-----------------+---+-------------------+-------------+----------+----------+----------+---------------+---------+-------------+
|NULL |SP-RIO PRETO NORTE|2023-07-28|3066     

In [ ]:
df_indicador_dev = df_indicador_dev.withColumn("data_nf", f.to_date(df_indicador_dev["data_nf"], "MM/dd/yyyy"))\
                .withColumn("data_dev", f.to_date(df_indicador_dev["data_dev"], "MM/dd/yyyy"))\
                .withColumn("data_lancamento_dev", f.to_date(df_indicador_dev["data_lancamento_dev"], "MM/dd/yyyy"))

In [ ]:
df_indicador_dev.createOrReplaceTempView("v_ind_dev")

In [ ]:
spark.sql(
    """
    SELECT id.*, CONCAT(id.cod_cliente, id.N_nf_saida) AS cl_nf
    FROM v_ind_dev id
    """
).show(1, False)

+-----+------------------+----------+----------+--------+-------+----------+-------------------+-----------+--------+----------------+-----------+-----------------------------------------+-----------------+---+-------------------+-------------+----------+----------+----------+---------------+---------+-------------+------------+
|Carga|Rota              |data_nf   |N_nf_saida|nota_dev|Valor  |data_dev  |data_lancamento_dev|Hora       |qtd_nota|Faturista       |cod_cliente|nome_cliente                             |tipo_dev         |cod|motivo_cancelamento|Representante|setor_resp|num_viagem|Fornecedor|nome_fornecedor|Motorista|placa_veiculo|cl_nf       |
+-----+------------------+----------+----------+--------+-------+----------+-------------------+-----------+--------+----------------+-----------+-----------------------------------------+-----------------+---+-------------------+-------------+----------+----------+----------+---------------+---------+-------------+------------+
|NULL |

In [ ]:
df_indicador_dev = spark.sql(
    """
    SELECT id.*, CONCAT(id.cod_cliente, id.N_nf_saida) AS cl_nf
    FROM v_ind_dev id
    """
)

In [ ]:
df_indicador_dev.createOrReplaceTempView("v_ind_dev")

In [ ]:
# Agora daqui pra frente vou trazer de volta a de vendas,
# para terminarmos de adicionar os dados, usando agora as novas
# tabelas que foram criadas.


spark.sql(
    """
    SELECT cr.*,
          CASE
            WHEN cr.cl_nf = id.cl_nf
              THEN cr.cl_nf
            ELSE "não tem"
          END AS Tem_Dev,
          CASE
            WHEN dr.cod_nf_valorPago = cr.cl_nf_valorPago
              THEN dr.cod_nf_valorPago
            ELSE "não abate dev"
          END AS esta_no_dev,
          CASE
            WHEN cr.cl_nf = id.cl_nf
              THEN id.tipo_dev
            ELSE "SEM DEV"
          END AS TIPO_DEV,
          COUNT(*) OVER (PARTITION BY cr.num_cr, cr.df_saida, valor_pago) AS Repetido,
          CASE
            WHEN cr.valor_pago = 0 AND cr.RepValida LIKE '%MAR E RIO%'
              THEN cr.valor_pago
            WHEN cr.valor_pago = 0 AND cr.RepValida NOT LIKE '%MAR E RIO%'
            AND cr.Comments ILIKE '%CUPOM PDV%'
              THEN cr.valor_baixado * 0.03
            WHEN cr.valor_pago = 0 AND cr.RepValida NOT LIKE '%MAR E RIO%'
            AND cr.Comments ILIKE '%RENEG%'
              THEN cr.valor_baixado * 0.015
            WHEN cr.valor_pago = 0 AND cr.RepValida NOT LIKE '%MAR E RIO%'
            AND cr.Comments NOT ILIKE '%RENEG%'
            AND cr.Comments NOT ILIKE '%CUPOM PDV%'
              THEN cr.`Comissão` * cr.valor_baixado
            ELSE cr.valor_pago
          END AS comissao_ajustada
    FROM v_com_rec cr
    LEFT JOIN v_ind_dev id ON id.cl_nf = cr.cl_nf
    LEFT JOIN v_dev_rec dr ON dr.cod_nf_valorPago = cr.cl_nf_valorPago

    """
).show(1, False)

+-------+--------------+-----------------------------------------+----------+--------+----------+-------------+--------+-------------+-------------+-------------------+--------------------+-------------------+--------------------+---------------+----------+--------------------------------------------------------------------------------------------------------------------------+---------------+----------+-----+---+-------+---------------+----------------------+--------------------+-------+-------------+--------+--------+-----------------+
|num_cr |codigo_cliente|nome_cliente                             |data_baixa|df_saida|NotaFiscal|valor_baixado|Comissão|parcela_atual|parcela_total|valor_pago_comissao|DocumentoRenegociado|comissao_paga_reneg|valor_pago_com_reneg|Representante  |data_nf   |Comments                                                                                                                  |RepValida      |valor_pago|Id   |REP|CORRETO|cl_nf          |cl_nf_valorBaix

In [ ]:
df_com_rec = spark.sql(
    """
    SELECT cr.*,
          CASE
            WHEN cr.cl_nf = id.cl_nf
              THEN cr.cl_nf
            ELSE "não tem"
          END AS Tem_Dev,
          CASE
            WHEN dr.cod_nf_valorPago = cr.cl_nf_valorPago
              THEN dr.cod_nf_valorPago
            ELSE "não abate dev"
          END AS esta_no_dev,
          CASE
            WHEN cr.cl_nf = id.cl_nf
              THEN id.tipo_dev
            ELSE "SEM DEV"
          END AS TIPO_DEV,
          COUNT(*) OVER (PARTITION BY cr.num_cr, cr.df_saida, valor_pago) AS Repetido,
          CASE
            WHEN cr.valor_pago = 0 AND cr.RepValida LIKE '%MAR E RIO%'
              THEN cr.valor_pago
            WHEN cr.valor_pago = 0 AND cr.RepValida NOT LIKE '%MAR E RIO%'
            AND cr.Comments ILIKE '%CUPOM PDV%'
              THEN cr.valor_baixado * 0.03
            WHEN cr.valor_pago = 0 AND cr.RepValida NOT LIKE '%MAR E RIO%'
            AND cr.Comments ILIKE '%RENEG%'
              THEN cr.valor_baixado * 0.015
            WHEN cr.valor_pago = 0 AND cr.RepValida NOT LIKE '%MAR E RIO%'
            AND cr.Comments NOT ILIKE '%RENEG%'
            AND cr.Comments NOT ILIKE '%CUPOM PDV%'
              THEN cr.`Comissão` * cr.valor_baixado
            ELSE cr.valor_pago
          END AS comissao_ajustada
    FROM v_com_rec cr
    LEFT JOIN v_ind_dev id ON id.cl_nf = cr.cl_nf
    LEFT JOIN v_dev_rec dr ON dr.cod_nf_valorPago = cr.cl_nf_valorPago

    """
)

In [ ]:
df_com_rec.createOrReplaceTempView("v_com_rec")

In [ ]:
spark.sql(
    """
    SELECT *
    FROM v_com_rec
    """
).show(1, False)

+-------+--------------+-----------------------------------------+----------+--------+----------+-------------+--------+-------------+-------------+-------------------+--------------------+-------------------+--------------------+---------------+----------+--------------------------------------------------------------------------------------------------------------------------+---------------+----------+-----+---+-------+---------------+----------------------+--------------------+-------+-------------+--------+--------+-----------------+
|num_cr |codigo_cliente|nome_cliente                             |data_baixa|df_saida|NotaFiscal|valor_baixado|Comissão|parcela_atual|parcela_total|valor_pago_comissao|DocumentoRenegociado|comissao_paga_reneg|valor_pago_com_reneg|Representante  |data_nf   |Comments                                                                                                                  |RepValida      |valor_pago|Id   |REP|CORRETO|cl_nf          |cl_nf_valorBaix

In [ ]:
df_com_rec.show(1, False)
df_dev_rec.show(1, False)
df_dev_rec_post.show(1, False)
df_indicador_dev.show(1, False)

+-------+--------------+-----------------------------------------+----------+--------+----------+-------------+--------+-------------+-------------+-------------------+--------------------+-------------------+--------------------+---------------+----------+--------------------------------------------------------------------------------------------------------------------------+---------------+----------+-----+---+-------+---------------+----------------------+--------------------+-------+-------------+--------+--------+-----------------+
|num_cr |codigo_cliente|nome_cliente                             |data_baixa|df_saida|NotaFiscal|valor_baixado|Comissão|parcela_atual|parcela_total|valor_pago_comissao|DocumentoRenegociado|comissao_paga_reneg|valor_pago_com_reneg|Representante  |data_nf   |Comments                                                                                                                  |RepValida      |valor_pago|Id   |REP|CORRETO|cl_nf          |cl_nf_valorBaix

In [ ]:
pdf1 = df_com_rec.toPandas()
pdf2 = df_dev_rec.toPandas()
pdf3 = df_dev_rec_post.toPandas()
pdf4 = df_indicador_dev.toPandas()

In [ ]:
# Aqui finalizamos a TempGeral onde se encontram os valores tratados. Após
# esse passo vamos estruturar tudo em uma base dos externos para que possamos
# validas os valores finais e ajustar o necessário para gerar os arquivos de cada
# representante individualmente.

with pd.ExcelWriter("/content/drive/MyDrive/externos_export/temp_geral_mar26.xlsx", engine="xlsxwriter") as writer:
  pdf1.to_excel(writer, sheet_name = "Geral-Recebidos-VD-MAR26", index=False)
  pdf2.to_excel(writer, sheet_name = "Geral-Recebidos-DEV-MAR26_", index=False)
  pdf3.to_excel(writer, sheet_name = "Geral-Recebidos-DEV-JAN26-POSTR", index=False)
  pdf4.to_excel(writer, sheet_name = "DEVS GERAL", index=False)

  workbook = writer.book
  worksheet1 = writer.sheets["Geral-Recebidos-VD-MAR26"]
  worksheet2 = writer.sheets["Geral-Recebidos-DEV-MAR26_"]
  worksheet3 = writer.sheets["Geral-Recebidos-DEV-JAN26-POSTR"]
  worksheet4 = writer.sheets["DEVS GERAL"]

  header_format = workbook.add_format({
      "bold":True,
      "text_wrap":True,
      "valign":"center",
      "fg_color":"#0255A0",
      "border":1
  })

  for col_num, value in enumerate(pdf1.columns.values):
        worksheet1.write(0, col_num, value, header_format)
        worksheet1.set_column(col_num, col_num, 25)

  for col_num, value in enumerate(pdf2.columns.values):
        worksheet2.write(0, col_num, value, header_format)
        worksheet2.set_column(col_num, col_num, 25)

  for col_num, value in enumerate(pdf3.columns.values):
        worksheet3.write(0, col_num, value, header_format)
        worksheet3.set_column(col_num, col_num, 25)

  for col_num, value in enumerate(pdf4.columns.values):
        worksheet4.write(0, col_num, value, header_format)
        worksheet4.set_column(col_num, col_num, 25)